# Projeto
O projeto Agroscope (IA) visa introduzir visão computacional para identificar padrões estruturais e texturas em folhas de plantas, de modo a identificar possíveis pragas.

## Importando Classes

In [ ]:
# Torch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import torchvision.datasets as datasets
from torch.utils.data import Subset, ConcatDataset, DataLoader

# Numpy
import numpy as np

# Matplotlib
import matplotlib.pyplot as plt

from PIL import Image

import random
import os

# Módulos Próprios
from utils.models import Get_Transform, TransformType, Pil_Loader

from dotenv import load_dotenv

In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)
print(".env loaded sucessfully" if load_dotenv() else "couldn't load .env")

# Definições

In [ ]:
NET = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

In [ ]:
# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCH = 8
BATCH_SIZE = 32
IMG_SIZE = 224
NUM_WORKERS = 8

# Learning Rates
LR_BACKBONE = 1e-4
LR_FC = 1e-3
WEIGHT_DECAY = 0.05

# Variáveis de Ambiente
MODEL_PATH = os.getenv("MODEL_PATH")
DATA_PATH = os.getenv("DATA_PATH")
EVAL_PATH = os.getenv("EVAL_PATH")
EXTRA_PATH = os.getenv("EXTRA_PATH")  # Folhas inteiras (mesmo tipo de foto do EVAL_PATH). Opcional.

print(f"Dispositivo Usado: {'GPU' if DEVICE.type == 'cuda' else 'CPU'}")

## Transform
`LeafCrop` recorta a folha em fotos com fundo claro antes do resize (em close-ups não faz nada). Veja `utils/models.py` e `utils/leaf.py`.

In [ ]:
transform_train = Get_Transform(TransformType.TRAIN, IMG_SIZE)
transform_eval = Get_Transform(TransformType.EVAL, IMG_SIZE)

In [ ]:
# Conferindo o recorte da folha em algumas imagens do EVAL_PATH (vermelho = região que vai para o modelo)
from PIL import ImageDraw
from utils.leaf import Leaf_BBox

preview = datasets.ImageFolder(EVAL_PATH, loader=Pil_Loader)
fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
for ax, i in zip(axes, random.sample(range(len(preview)), 6)):
    img = preview.loader(preview.samples[i][0])
    bbox = Leaf_BBox(img)
    if bbox is not None:
        ImageDraw.Draw(img).rectangle(bbox, outline="red", width=max(3, img.width // 100))
    ax.imshow(img); ax.set_title(preview.classes[preview.samples[i][1]]); ax.axis("off")
plt.tight_layout()
plt.show()

## Dataset Principal
Dois `ImageFolder` sobre a mesma pasta, um com o transform de treino e outro com o de avaliação. Assim cada split usa o transform certo (antes, trocar `val_dataset.dataset.transform` não tinha efeito nenhum).

In [ ]:
dataset = datasets.ImageFolder(root=DATA_PATH, transform=transform_train, loader=Pil_Loader)
dataset_eval_view = datasets.ImageFolder(root=DATA_PATH, transform=transform_eval, loader=Pil_Loader)
num_labels = len(dataset.classes)

In [ ]:
def Balance(samples, num_labels :int, value :int=None):
    """Agrupa índices por classe e devolve o limite por classe (menor classe, ou `value` se for menor)."""
    class_indices = {label: [] for label in range(num_labels)}
    for idx, (_, label) in enumerate(samples):
        class_indices[label].append(idx)

    low = min(len(imgs) for imgs in class_indices.values())
    if value is not None:
        low = min(low, value)
    return class_indices, low


class_indices, low = Balance(dataset.samples, num_labels, 1200)

In [ ]:
balanced_indices = []

for class_label, indices in class_indices.items():
    random.shuffle(indices)
    selected_indices = indices[:low]
    balanced_indices.extend(selected_indices)
    print(f"Class {class_label}: {len(selected_indices)}")

random.shuffle(balanced_indices)

In [ ]:
dataset_len = len(balanced_indices)

train_size = int(0.7 * dataset_len)
val_size = int(0.15 * dataset_len)
test_size = dataset_len - train_size - val_size

print(f"All Dataset: {dataset_len}")
print(f"Train (70%): {train_size}")
print(f"Eval (15%): {val_size}")
print(f"Test (15%): {test_size}")

In [ ]:
perm = torch.randperm(dataset_len, generator=torch.Generator().manual_seed(SEED)).tolist()
train_idx = [balanced_indices[i] for i in perm[:train_size]]
val_idx = [balanced_indices[i] for i in perm[train_size:train_size + val_size]]
test_idx = [balanced_indices[i] for i in perm[train_size + val_size:]]

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset_eval_view, val_idx)
test_dataset = Subset(dataset_eval_view, test_idx)

## Dataset complementar (folhas inteiras)
O `DATA_PATH` só tem close-ups; sem exemplos de folha inteira o modelo não generaliza para o tipo de foto do `EVAL_PATH` (32% de acerto). O `EXTRA_PATH` traz fotos do mesmo tipo do teste.

**Para não vazar o teste para o treino**, qualquer arquivo com o mesmo nome de um arquivo do `EVAL_PATH` é ignorado. Arquivos corrompidos também são pulados.

In [ ]:
def Is_Valid_Extra(path :str) -> bool:
    name = os.path.basename(path)
    if not name.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
        return False
    if name in eval_file_names:
        return False
    try:
        with Image.open(path) as img:
            img.draft("RGB", (256, 256))
            img.load()
        return True
    except Exception as e:
        print(f"Ignorando arquivo corrompido: {path} ({e})")
        return False


extra_train_dataset, extra_val_dataset = None, None

if EXTRA_PATH:
    eval_file_names = {os.path.basename(p) for p, _ in datasets.ImageFolder(EVAL_PATH).samples}

    extra_train_view = datasets.ImageFolder(EXTRA_PATH, transform=transform_train, loader=Pil_Loader, is_valid_file=Is_Valid_Extra)
    extra_eval_view = datasets.ImageFolder(EXTRA_PATH, transform=transform_eval, loader=Pil_Loader, is_valid_file=Is_Valid_Extra)
    assert extra_train_view.classes == dataset.classes, f"Classes diferentes: {extra_train_view.classes} vs {dataset.classes}"

    extra_perm = torch.randperm(len(extra_train_view), generator=torch.Generator().manual_seed(SEED)).tolist()
    extra_cut = int(0.8 * len(extra_perm))
    extra_train_dataset = Subset(extra_train_view, extra_perm[:extra_cut])
    extra_val_dataset = Subset(extra_eval_view, extra_perm[extra_cut:])

    train_dataset = ConcatDataset([train_dataset, extra_train_dataset])
    val_dataset = ConcatDataset([val_dataset, extra_val_dataset])
    print(f"Extra -> train={len(extra_train_dataset)} val={len(extra_val_dataset)} (arquivos do EVAL_PATH excluídos)")
else:
    print("EXTRA_PATH não definido: treinando só com close-ups (espere baixa acurácia em fotos de folha inteira)")

In [ ]:
make_loader = lambda ds, shuffle: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
                                            pin_memory=True, persistent_workers=True)

train_loader = make_loader(train_dataset, True)
val_loader = make_loader(val_dataset, False)
test_loader = make_loader(test_dataset, False)

print("Classes:", dataset.classes)
print(f"Split -> train={len(train_dataset)} val={len(val_dataset)} test={len(test_dataset)}")

# Modelo
Usaremos um modelo já treinado com entendimento de padrões para podermos realizar um transfer-learning e aprimora-lo com dados próprios.

In [ ]:
in_features = NET.classifier[2].in_features
NET.classifier[2] = nn.Linear(in_features, num_labels)

In [ ]:
# Descongelando apenas os últimos estágios do ConvNeXt (features.5 = estágio 3, features.7 = estágio 4)
for param in NET.parameters():
    param.requires_grad = False

# Palavras chaves para ativação de gradiente
keywords = ["classifier", "features.5", "features.6", "features.7"]

for name, param in NET.named_parameters():
    if any(k in name for k in keywords):
        param.requires_grad = True

trainable = sum(p.numel() for p in NET.parameters() if p.requires_grad)
print(f"Parâmetros treináveis: {trainable:,} / {sum(p.numel() for p in NET.parameters()):,}")

In [ ]:
backbone_params = [p for n, p in NET.named_parameters() if p.requires_grad and "classifier" not in n]
head_params = [p for n, p in NET.named_parameters() if p.requires_grad and "classifier" in n]

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": LR_BACKBONE},
    {"params": head_params, "lr": LR_FC},
], weight_decay=WEIGHT_DECAY)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Scheduler - warmup linear de meia época + cosseno até zero, avançando a cada batch.
# (Antes: CosineAnnealingLR(T_max=EPOCH) com step por batch -> LR oscilava a cada 3 batches)
total_steps = EPOCH * len(train_loader)
warmup_steps = max(1, len(train_loader) // 2)

def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Treinando o modelo

In [ ]:
NET.to(DEVICE)
use_amp = DEVICE.type == "cuda"

# Historico para diagnostico de overfitting (train vs val por epoca)
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc, best_state = -1.0, None

for epoch in range(EPOCH):
    running_loss = .0
    running_correct = 0
    NET.train()

    for images, labels in train_loader:
        inputs = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=use_amp):
            outputs = NET(inputs)
            loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item() * inputs.size(0)
        running_correct += (outputs.argmax(dim=1) == labels).sum().item()

    train_loss = running_loss / len(train_dataset)
    train_acc = running_correct / len(train_dataset)

    NET.eval()
    classified_right = 0
    running_val_loss = .0

    with torch.no_grad(), torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=use_amp):
        for images, labels in val_loader:
            inputs = images.to(DEVICE)
            labels = labels.to(DEVICE)
            outputs = NET(inputs)

            running_val_loss += criterion(outputs, labels).item() * inputs.size(0)
            classified_right += (outputs.argmax(dim=1) == labels).sum().item()

    val_accuracy = classified_right / len(val_dataset)
    val_loss = running_val_loss / len(val_dataset)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_accuracy)

    # Guarda os pesos da melhor época na validação
    if val_accuracy >= best_val_acc:
        best_val_acc = val_accuracy
        best_state = {k: v.detach().clone() for k, v in NET.state_dict().items()}

    print(f"Epoch {epoch + 1} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | val_acc: {val_accuracy:.4f}")

NET.load_state_dict(best_state)
print(f"Melhor val_acc: {best_val_acc:.4f}")

## Diagnóstico de Overfitting (curvas de treino vs. validação)
Como ler o gráfico abaixo:
- **Loss de treino caindo e loss de validação subindo (ou estagnada)** = overfitting claro. A partir da época em que a linha de validação vira pra cima, o modelo está decorando o treino.
- **As duas caindo juntas, com um gap pequeno e estável** = sem overfitting relevante; se a acurácia ainda está baixa, é sinal de underfitting (falta capacidade/dado), não excesso.
- **Gap grande entre acurácia de treino e validação** (ex. treino 98%, val 80%) mesmo com as losses ainda caindo = overfitting já instalado, mesmo que a loss de validação não tenha subido ainda — geralmente o próximo sinal.

Obs.: com label smoothing a loss mínima não é 0, e o treino usa augmentation forte — por isso a acurácia de treino pode ficar abaixo da de validação.

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(epochs_range, history["train_loss"], label="Train Loss", marker="o")
axes[0].plot(epochs_range, history["val_loss"], label="Val Loss", marker="o")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss: Treino vs. Validação")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["train_acc"], label="Train Acc", marker="o")
axes[1].plot(epochs_range, history["val_acc"], label="Val Acc", marker="o")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Acurácia")
axes[1].set_title("Acurácia: Treino vs. Validação")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

gap = history["train_acc"][-1] - history["val_acc"][-1]
print(f"Gap final (train_acc - val_acc): {gap:.4f}")

# Avaliando o modelo
Métricas de avaliação avançadas devem ser aplicadas para validar o resultado, como visto na etapa 1.

In [ ]:
# Avaliação de resultados
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score

In [ ]:
def Evaluate(loader):
    NET.eval()
    total_loss, y_true, y_pred = 0.0, [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = NET(x)
            total_loss += criterion(logits, y).item() * x.size(0)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(logits.argmax(dim=1).cpu().numpy())
    return total_loss / len(loader.dataset), np.array(y_true), np.array(y_pred)


avg_loss, y_true, y_pred = Evaluate(test_loader)
test_acc = accuracy_score(y_true, y_pred)

print(f"""
avg_loss: {avg_loss}
acc: {test_acc}
f1m: {f1_score(y_true, y_pred, average="macro")}
bacc: {balanced_accuracy_score(y_true, y_pred)}
""")

# Teste Manual
Fonte externa (`EVAL_PATH`): fotos de folha inteira, nunca vistas no treino.

__logits__: Valores brutos (ou pontuações) gerados pela última camada linear de uma rede neural antes de passarem por uma função de ativação, como softmax ou sigmóide

In [ ]:
dataset_eval = datasets.ImageFolder(root=EVAL_PATH, transform=transform_eval, loader=Pil_Loader)

# Garante que os indices de classe do dataset de avaliacao externa
# batem exatamente com os indices usados no treino (evita 'inversao' silenciosa
# caso as pastas do EVAL_PATH estejam em ordem/alfabeto diferente ou faltando classes)
if dataset_eval.classes != dataset.classes:
    print(f"Aviso: ordem de classes diferente entre treino {dataset.classes} e eval {dataset_eval.classes}")
    print("Remapeando os targets do eval para o class_to_idx do treino...")
    dataset_eval.samples = [
        (path, dataset.class_to_idx[dataset_eval.classes[old_idx]])
        for path, old_idx in dataset_eval.samples
    ]
    dataset_eval.targets = [s[1] for s in dataset_eval.samples]
    dataset_eval.class_to_idx = dataset.class_to_idx
else:
    print("Classes alinhadas entre treino e eval:", dataset.classes)

# Mapeamento indice -> nome da classe (usar sempre o do dataset de treino, que é o canônico)
idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}
print("idx_to_class:", idx_to_class)

eval_loader = DataLoader(dataset_eval, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)

In [ ]:
avg_loss, y_true, y_pred = Evaluate(eval_loader)
eval_acc = accuracy_score(y_true, y_pred)

print("METRICS")
print(f"""
avg_loss: {avg_loss}
acc: {eval_acc}
f1m: {f1_score(y_true, y_pred, average="macro")}
bacc: {balanced_accuracy_score(y_true, y_pred)}
""")
print(f"Accuracy of the model on the external set: {eval_acc * 100:.2f}% ({len(dataset_eval)} imagens)")

# Visualização Detalhada dos Resultados
Matriz de confusão e relatório por classe (precision/recall/f1), usando os nomes reais das classes em vez de índices numéricos — facilita ver quais doenças estão sendo confundidas entre si.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

class_names = [idx_to_class[i] for i in range(len(idx_to_class))]

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))

fig, ax = plt.subplots(figsize=(7, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
ax.set_title("Matriz de Confusão - Teste Manual (fonte externa)")
plt.tight_layout()
plt.show()

In [ ]:
# Relatório por classe: precision, recall, f1-score e suporte (n de amostras) por classe.
# 'support' importa muito aqui: com poucas amostras por classe, olhe o número junto da métrica,
# não só a porcentagem isolada.
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

## Grad-CAM — Onde o modelo está "olhando"
Gera um mapa de calor sobre a imagem mostrando quais regiões mais influenciaram a predição da classe escolhida. Útil para diagnosticar se erros vêm de um atalho (ex. cor/textura genérica) ou de uma região irrelevante da imagem (fundo, sombra), em vez da lesão real.

**Correção importante**: a versão anterior registrava os hooks uma vez só e chamava `remove_hooks()` no fim da primeira célula. As células seguintes continuavam usando o mesmo objeto, mas sem hooks as ativações/gradientes não eram mais atualizados — todo heatmap era uma cópia do último calculado. Agora os hooks são registrados e removidos **dentro de cada chamada**, então não existe estado velho.

In [ ]:
class GradCAM:
    """Grad-CAM via hooks registrados a cada chamada (sem estado entre imagens).
    Para o ConvNeXt, o alvo natural é NET.features[-1] (último estágio,
    ainda com dimensão espacial, antes do pooling global)."""

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

    def __call__(self, input_tensor, class_idx=None):
        store = {}

        def forward_hook(module, inp, output):
            store["activations"] = output
            # Hook no tensor de saída: pega o gradiente sem precisar de full_backward_hook
            output.register_hook(lambda grad: store.__setitem__("gradients", grad))

        handle = self.target_layer.register_forward_hook(forward_hook)
        try:
            self.model.eval()
            with torch.enable_grad():
                output = self.model(input_tensor)
                if class_idx is None:
                    class_idx = output.argmax(dim=1).item()
                self.model.zero_grad()
                output[:, class_idx].backward()
        finally:
            handle.remove()

        # Peso de cada canal = media espacial do gradiente (GAP)
        weights = store["gradients"].mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * store["activations"]).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx


def tensor_to_image(img_tensor):
    """Desfaz a normalizacao do ImageNet para exibir a imagem original."""
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img_tensor.detach().cpu().numpy().transpose(1, 2, 0)
    img = std * img + mean
    return np.clip(img, 0, 1)


def show_cams(items, max_rows :int=12):
    """items: lista de (img_tensor, cam, true_idx, pred_idx)."""
    items = items[:max_rows]
    if not items:
        return
    fig, axes = plt.subplots(len(items), 2, figsize=(8, 4 * len(items)), squeeze=False)
    for row, (img_tensor, cam, true_idx, pred_idx) in enumerate(items):
        img = tensor_to_image(img_tensor)
        axes[row, 0].imshow(img)
        axes[row, 0].set_title(f"Original — Real: {idx_to_class[true_idx]}")
        axes[row, 0].axis("off")
        axes[row, 1].imshow(img)
        axes[row, 1].imshow(cam, cmap="jet", alpha=0.5)
        axes[row, 1].set_title(f"Grad-CAM — Previsto: {idx_to_class[pred_idx]}")
        axes[row, 1].axis("off")
    plt.tight_layout()
    plt.show()


NET.to(DEVICE)
gradcam = GradCAM(NET, NET.features[-1])

# Roda o Grad-CAM uma única vez em todo o eval_loader e separa acertos e erros
errors, corrects = [], []
for images, labels in eval_loader:
    images = images.to(DEVICE)
    for i in range(images.size(0)):
        cam, pred_idx = gradcam(images[i:i + 1])  # usa a predicao do modelo
        true_idx = labels[i].item()
        (corrects if pred_idx == true_idx else errors).append((images[i].cpu(), cam, true_idx, pred_idx))

print(f"{len(errors)} erro(s) e {len(corrects)} acerto(s) no eval_loader")

# Grad-CAM

## Incorrect

In [ ]:
show_cams(errors)

## Correct

In [ ]:
# Amostra aleatória de acertos (todas as classes)
show_cams(random.sample(corrects, min(12, len(corrects))))

### Teste de sanidade: o heatmap depende do conteúdo?
Dois testes:
1. **Imagens sem folha** (ruído e cinza liso): o heatmap delas não deve se parecer com o das folhas reais.
2. **Centro de massa do heatmap** em todas as imagens do eval: se o modelo estivesse olhando sempre para o mesmo ponto, o desvio padrão do centro de massa seria ~0. Um desvio de vários pixels mostra que o foco acompanha a imagem (lesões, bordas, nervuras), não uma posição fixa.

In [ ]:
input_shape = dataset_eval[0][0].shape  # (C, H, W)

noise_tensor = torch.randn(1, *input_shape).to(DEVICE)
cam_noise, pred_noise = gradcam(noise_tensor)

gray_tensor = torch.zeros(1, *input_shape).to(DEVICE)
cam_gray, pred_gray = gradcam(gray_tensor)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, t, cam, pred, title in [(axes[0], noise_tensor, cam_noise, pred_noise, "Ruído aleatório"),
                                (axes[1], gray_tensor, cam_gray, pred_gray, "Cinza liso")]:
    ax.imshow(tensor_to_image(t[0]))
    ax.imshow(cam, cmap="jet", alpha=0.5)
    ax.set_title(f"{title} -> previsto: {idx_to_class[pred]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

# Centro de massa de cada heatmap (em pixels, imagem de IMG_SIZE x IMG_SIZE)
yy, xx = np.mgrid[:IMG_SIZE, :IMG_SIZE]
centers = np.array([((cam * xx).sum() / cam.sum(), (cam * yy).sum() / cam.sum()) for _, cam, _, _ in errors + corrects])
print(f"Centro de massa médio (x, y): {centers.mean(0).round(1)}  |  desvio padrão: {centers.std(0).round(1)}")

plt.figure(figsize=(4.5, 4.5))
plt.scatter(centers[:, 0], centers[:, 1], s=8, alpha=0.5)
plt.xlim(0, IMG_SIZE); plt.ylim(IMG_SIZE, 0)
plt.title("Centro de massa do Grad-CAM por imagem")
plt.grid(alpha=0.3)
plt.show()

## Test-Time Augmentation (TTA)
Roda cada imagem de avaliação em algumas variações (original, flips), tira a média das probabilidades e só então decide a classe. Não muda o modelo nem o treino — só estabiliza a predição na hora da inferência.

In [ ]:
def predict_tta(model, images):
    """Média das probabilidades (softmax) da imagem original + flip horizontal + flip vertical.
    images já deve estar normalizado (mesmo transform_eval) e no DEVICE."""
    model.eval()
    with torch.no_grad():
        probs = sum(F.softmax(model(v), dim=1) for v in (images, torch.flip(images, dims=[3]), torch.flip(images, dims=[2])))
    return probs / 3


y_true_tta, y_pred_tta = [], []

for images, labels in eval_loader:
    probs = predict_tta(NET, images.to(DEVICE))
    y_true_tta.extend(labels.numpy())
    y_pred_tta.extend(probs.argmax(dim=1).cpu().numpy())

acc_tta = accuracy_score(y_true_tta, y_pred_tta)
print(f"Accuracy com TTA: {acc_tta * 100:.2f}%  (sem TTA: {eval_acc * 100:.2f}%)")

# Salvando Modelo
Atenção: para usar o modelo em produção, a inferência precisa aplicar o **mesmo** `transform_eval` (incluindo o `LeafCrop`), senão o problema de foco no centro volta.

In [ ]:
import datetime as dt

def Save(path :str=MODEL_PATH):
    t = dt.datetime.now()  # (antes era argumento default -> ficava fixo no horário em que a célula foi definida)
    os.makedirs(os.path.dirname(path), exist_ok=True)

    torch.save({
        "model_state_dict": NET.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "class_names": dataset.classes,
        "num_classes": len(dataset.classes),
        "model_info": {
            "version": "2.0",
            "architecture": "convnext_tiny",
            "training_date": t.strftime("%d-%m-%Y %H:%M:%S"),
            "img_size": IMG_SIZE,
            "preprocessing": "LeafCrop + Resize((img_size, img_size)) + Normalize(ImageNet)",
            "best_val_accuracy": best_val_acc,
            "test_accuracy": test_acc,
            "external_accuracy": eval_acc,
        }
    }, path)
    print(f"Modelo salvo em {path}")

Save()